# keyboard.actions

> Keyboard navigation focus zone and action factories for the virtual collection.

In [ ]:
#| default_exp keyboard.actions

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Dict, Optional, Tuple

from cjm_fasthtml_keyboard_navigation.core.focus_zone import FocusZone
from cjm_fasthtml_keyboard_navigation.core.actions import KeyAction
from cjm_fasthtml_keyboard_navigation.core.navigation import ScrollOnly

from cjm_fasthtml_virtual_collection.core.html_ids import VirtualCollectionHtmlIds
from cjm_fasthtml_virtual_collection.core.button_ids import VirtualCollectionButtonIds
from cjm_fasthtml_virtual_collection.core.models import VirtualCollectionUrls

## Focus Zone

In [ ]:
#| export
def create_collection_focus_zone(
    ids: VirtualCollectionHtmlIds,  # HTML IDs for this collection instance
    hidden_input_prefix: Optional[str] = None,  # Prefix for keyboard nav hidden inputs
) -> FocusZone:  # Configured focus zone for the collection
    """Create a focus zone for a virtual collection viewport."""
    return FocusZone(
        id=ids.collection,
        item_selector=None,
        navigation=ScrollOnly(),
        zone_focus_classes=(),
        item_focus_classes=(),
        hidden_input_prefix=hidden_input_prefix or f"{ids.prefix}-vc",
    )

## Navigation Actions

In [ ]:
#| export
def create_collection_nav_actions(
    zone_id: str,  # Focus zone ID to restrict actions to
    button_ids: VirtualCollectionButtonIds,  # Button IDs for HTMX triggers
    disable_in_modes: Tuple[str, ...] = (),  # Mode names that disable navigation
) -> Tuple[KeyAction, ...]:  # Standard collection navigation actions
    """Create standard keyboard navigation actions for a virtual collection."""
    zone_ids = (zone_id,)
    not_modes = disable_in_modes if disable_in_modes else ()

    return (
        # --- Row navigation ---
        KeyAction(
            key="ArrowUp",
            htmx_trigger=button_ids.nav_up,
            zone_ids=zone_ids,
            not_modes=not_modes,
            description="Previous row",
            hint_group="Navigation",
        ),
        KeyAction(
            key="ArrowDown",
            htmx_trigger=button_ids.nav_down,
            zone_ids=zone_ids,
            not_modes=not_modes,
            description="Next row",
            hint_group="Navigation",
        ),

        # --- Page jump ---
        KeyAction(
            key="PageUp",
            htmx_trigger=button_ids.nav_page_up,
            zone_ids=zone_ids,
            not_modes=not_modes,
            description="Page up",
            hint_group="Navigation",
        ),
        KeyAction(
            key="PageDown",
            htmx_trigger=button_ids.nav_page_down,
            zone_ids=zone_ids,
            not_modes=not_modes,
            description="Page down",
            hint_group="Navigation",
        ),

        # --- First/last ---
        KeyAction(
            key="Home",
            htmx_trigger=button_ids.nav_first,
            zone_ids=zone_ids,
            not_modes=not_modes,
            description="First row",
            hint_group="Navigation",
        ),
        KeyAction(
            key="End",
            htmx_trigger=button_ids.nav_last,
            zone_ids=zone_ids,
            not_modes=not_modes,
            description="Last row",
            hint_group="Navigation",
        ),
    )

## URL Map Builder

In [ ]:
#| export
def build_collection_url_map(
    button_ids: VirtualCollectionButtonIds,  # Button IDs for this collection
    urls: VirtualCollectionUrls,  # URL bundle for routing
) -> Dict[str, str]:  # Mapping of button ID -> route URL
    """Build url_map for render_keyboard_system with all collection navigation buttons."""
    return {
        button_ids.nav_up: urls.nav_up,
        button_ids.nav_down: urls.nav_down,
        button_ids.nav_page_up: urls.nav_page_up,
        button_ids.nav_page_down: urls.nav_page_down,
        button_ids.nav_first: urls.nav_first,
        button_ids.nav_last: urls.nav_last,
    }

## Tests

In [ ]:
from cjm_fasthtml_virtual_collection.core.html_ids import VirtualCollectionHtmlIds
from cjm_fasthtml_virtual_collection.core.button_ids import VirtualCollectionButtonIds
from cjm_fasthtml_virtual_collection.core.models import VirtualCollectionUrls

ids = VirtualCollectionHtmlIds(prefix="t")
btn_ids = VirtualCollectionButtonIds(prefix="t")
urls = VirtualCollectionUrls(
    nav_up="/nav_up", nav_down="/nav_down",
    nav_page_up="/nav_page_up", nav_page_down="/nav_page_down",
    nav_first="/nav_first", nav_last="/nav_last",
)

In [ ]:
# Test focus zone
zone = create_collection_focus_zone(ids)
assert zone.id == "t-collection"
assert zone.item_selector is None
assert zone.zone_focus_classes == ()
print(f"Zone: id={zone.id}, nav={zone.navigation}")

Zone: id=t-collection, nav=ScrollOnly()


In [ ]:
# Test nav actions
actions = create_collection_nav_actions(zone.id, btn_ids)
assert len(actions) == 6

keys = [a.key for a in actions]
assert "ArrowUp" in keys
assert "ArrowDown" in keys
assert "PageUp" in keys
assert "PageDown" in keys
assert "Home" in keys
assert "End" in keys

# All actions should use htmx_trigger (not js_callback)
for action in actions:
    assert action.htmx_trigger is not None
    assert action.js_callback is None

print(f"Nav actions: {len(actions)} actions, all htmx_trigger")
for a in actions:
    print(f"  {a.key} -> {a.htmx_trigger}")

Nav actions: 6 actions, all htmx_trigger
  ArrowUp -> t-btn-nav-up
  ArrowDown -> t-btn-nav-down
  PageUp -> t-btn-nav-page-up
  PageDown -> t-btn-nav-page-down
  Home -> t-btn-nav-first
  End -> t-btn-nav-last


In [ ]:
# Test URL map
url_map = build_collection_url_map(btn_ids, urls)
assert len(url_map) == 6
assert url_map[btn_ids.nav_up] == "/nav_up"
assert url_map[btn_ids.nav_down] == "/nav_down"
assert url_map[btn_ids.nav_first] == "/nav_first"
assert url_map[btn_ids.nav_last] == "/nav_last"
print(f"URL map: {len(url_map)} entries")

URL map: 6 entries


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()